In [1]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import sys

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

print("Project root:", project_root)
print("Python executable:", sys.executable)

Project root: c:\Users\paula\Documents\Cours\A4\S2\WebData\Project\f1_semantic_web_project
Python executable: c:\Users\paula\Documents\Cours\A4\S2\WebData\Project\f1_semantic_web_project\.venv\Scripts\python.exe


In [3]:
import pandas as pd

entity_df = pd.read_csv("../data/processed/entity_catalog.csv")
relation_df = pd.read_csv("../data/processed/relation_catalog.csv")

print(f"Entities: {len(entity_df)}")
print(f"Relations: {len(relation_df)}")

Entities: 3265
Relations: 375


In [4]:
from src.kg.rdf_builder import build_rdf_graph

graph = build_rdf_graph(entity_df, relation_df)
print("Number of triples:", len(graph))

Number of triples: 2441


In [5]:
from pathlib import Path

Path("../kg_artifacts").mkdir(exist_ok=True)

graph.serialize("../kg_artifacts/initial_graph.ttl", format="turtle")
print("Sauvegardé → kg_artifacts/initial_graph.ttl")

Sauvegardé → kg_artifacts/initial_graph.ttl


In [13]:
from src.kg.wikidata_linker import link_entities

# test sur les équipes seulement
teams = entity_df[entity_df["entity_type"] == "Team"][["entity_text", "entity_type"]].drop_duplicates().to_dict("records")
print(f"Test sur {len(teams)} équipes...")

linked_teams = link_entities(teams)

for r in linked_teams:
    status = r["wikidata_qid"] or "NOT FOUND"
    print(f"{r['entity_text']:25} | {status:12} | {r['confidence']} | {r['wikidata_description']}")

Test sur 10 équipes...
Alpine                    | Q98930155    | 0.8 | French Formula One racing team
Aston Martin              | Q104620704   | 0.8 | Formula One racing team
Audi                      | Q120756696   | 0.8 | Formula One team
Cadillac                  | Q131323328   | 0.8 | Formula One team and constructor
Ferrari                   | Q2009683     | 0.8 | Ferrari F1-2000 Formula 1 car
Haas F1 Team              | Q16300222    | 1.0 | Formula 1 team
McLaren                   | Q172030      | 1.0 | British Formula One team
Mercedes                  | Q172721      | 0.8 | auto racing team
Red Bull Racing           | Q173663      | 1.0 | Formula One racing team based in Milton Keynes, England
Williams                  | Q171337      | 0.8 | British Formula One team and constructor


In [12]:
from src.kg.wikidata_linker import search_wikidata_entity

# test avec nom enrichi
for name in ["Alpine F1 Team", "Audi F1", "Cadillac F1", "Mercedes F1"]:
    result = search_wikidata_entity(name, "Team")
    if result:
        print(f"{name:25} | {result['qid']:12} | {result['confidence']} | {result['description']}")
    else:
        print(f"{name:25} | NOT FOUND")

Alpine F1 Team            | Q98930155    | 1.0 | French Formula One racing team
Audi F1                   | Q120756696   | 0.8 | Formula One team
Cadillac F1               | Q131323328   | 0.6 | Formula One team and constructor
Mercedes F1               | Q172721      | 0.8 | auto racing team


In [19]:
linked_teams = link_entities(teams)

for r in linked_teams:
    status = r["wikidata_qid"] or "NOT FOUND"
    print(f"{r['entity_text']:25} | {status:12} | {r['confidence']} | {r['wikidata_description']}")

Alpine                    | Q98930155    | 1.0 | French Formula One racing team
Aston Martin              | Q104620704   | 0.8 | Formula One racing team
Audi                      | Q120756696   | 1.0 | Formula One team
Cadillac                  | Q131323328   | 0.6 | Formula One team and constructor
Ferrari                   | Q169898      | 1.0 | Formula One team
Haas F1 Team              | Q16300222    | 1.0 | Formula 1 team
McLaren                   | Q172030      | 1.0 | British Formula One team
Mercedes                  | Q172721      | 1.0 | auto racing team
Red Bull Racing           | Q173663      | 1.0 | Formula One racing team based in Milton Keynes, England
Williams                  | Q171337      | 0.8 | British Formula One team and constructor


In [10]:
import requests

WIKIDATA_SEARCH_URL = "https://www.wikidata.org/w/api.php"
HEADERS = {"User-Agent": "F1-KG-Project/1.0 (student project; paula@example.com)"}

params = {
    "action": "wbsearchentities",
    "search": "Scuderia Ferrari",
    "language": "en",
    "format": "json",
    "limit": 5,
}
response = requests.get(WIKIDATA_SEARCH_URL, params=params, headers=HEADERS, timeout=10)
for r in response.json()["search"]:
    print(r["id"], "|", r["label"], "|", r.get("description", ""))

Q169898 | Scuderia Ferrari | Formula One team
Q110930131 | Scuderia Ferrari Race 2013 | 2013 video game
Q5866426 | history of Scuderia Ferrari | aspect of history
Q2274853 | Ferrari Grand Prix results | Wikimedia list article


In [14]:
from src.kg.wikidata_linker import search_wikidata_entity
result = search_wikidata_entity("Ferrari", "Team")
print(result)

{'uri': 'http://www.wikidata.org/entity/Q2009683', 'qid': 'Q2009683', 'label': 'Ferrari F1-2000', 'description': 'Ferrari F1-2000 Formula 1 car', 'confidence': 0.8}


In [18]:
for search in ["Cadillac Formula One", "Cadillac F1", "Mercedes AMG F1", "Mercedes F1 team"]:
    params = {
        "action": "wbsearchentities",
        "search": search,
        "language": "en",
        "format": "json",
        "limit": 3,
    }
    response = requests.get(WIKIDATA_SEARCH_URL, params=params, headers=HEADERS, timeout=10)
    results = response.json().get("search", [])
    print(f"\n{search}:")
    for r in results:
        print(f"  {r['id']} | {r['label']} | {r.get('description', '')}")


Cadillac Formula One:

Cadillac F1:
  Q131323328 | Cadillac Formula 1 Team | Formula One team and constructor

Mercedes AMG F1:
  Q173358 | Mercedes AMG High Performance Powertrains | British Formula One engine manufacturer
  Q28450077 | Mercedes AMG F1 W08 EQ Power+ | Mercedes AMG Petronas F1 Team 2017 Formula 1 car
  Q48965018 | Mercedes AMG F1 W09 EQ Power+ | Mercedes-AMG Petronas Motorsport 2018 Formula 1 car

Mercedes F1 team:
  Q172721 | Mercedes F1 Team | auto racing team


In [20]:
# test sur les drivers principaux seulement
key_drivers = [
    "Lewis Hamilton", "Max Verstappen", "Michael Schumacher",
    "Ayrton Senna", "Fernando Alonso", "Sebastian Vettel",
    "Charles Leclerc", "Lando Norris", "George Russell"
]

driver_rows = [{"entity_text": name, "entity_type": "Driver"} for name in key_drivers]
linked_drivers = link_entities(driver_rows)

for r in linked_drivers:
    status = r["wikidata_qid"] or "NOT FOUND"
    print(f"{r['entity_text']:25} | {status:12} | {r['confidence']} | {r['wikidata_description']}")

Lewis Hamilton            | Q9673        | 1.0 | British racing driver
Max Verstappen            | Q2239218     | 1.0 | Dutch and Belgian racing driver
Michael Schumacher        | Q9671        | 1.0 | German racing driver
Ayrton Senna              | Q10490       | 1.0 | Brazilian racing driver (1960-1994)
Fernando Alonso           | Q10514       | 1.0 | Spanish racing driver
Sebastian Vettel          | Q42311       | 1.0 | German racing driver
Charles Leclerc           | Q17541912    | 1.0 | Monegasque racing driver
Lando Norris              | Q22007193    | 1.0 | British and Belgian racing driver (born 1999)
George Russell            | Q17319645    | 1.0 | British racing driver (born 1998)


In [21]:
# on garde seulement les types qu'on sait bien gérer
types_to_link = ["Driver", "Team", "GrandPrix", "Season"]

entities_to_link = (
    entity_df[entity_df["entity_type"].isin(types_to_link)]
    [["entity_text", "entity_type"]]
    .drop_duplicates()
    .to_dict("records")
)

print(f"Entités à linker: {len(entities_to_link)}")

Entités à linker: 939


In [22]:
# on garde seulement les entités avec au moins 2 mots pour Driver
# et on exclut les entités qui contiennent des chiffres seuls
def is_worth_linking(row):
    text = row["entity_text"]
    entity_type = row["entity_type"]
    
    # exclut les textes trop courts
    if len(text) < 4:
        return False
    # exclut les textes qui commencent par un chiffre
    if text[0].isdigit():
        return False
    return True

entities_filtered = [r for r in entities_to_link if is_worth_linking(r)]
print(f"Après filtrage: {len(entities_filtered)} entités")

Après filtrage: 840 entités


In [23]:
from src.kg.wikidata_linker import link_entities

print("Linking en cours... (peut prendre 4-5 minutes)")
linked = link_entities(entities_filtered, delay=0.3)

# stats rapides
found = [r for r in linked if r["wikidata_uri"]]
not_found = [r for r in linked if not r["wikidata_uri"]]

print(f"\nRésultats:")
print(f"  Trouvés    : {len(found)}")
print(f"  Non trouvés: {len(not_found)}")

Linking en cours... (peut prendre 4-5 minutes)

Résultats:
  Trouvés    : 475
  Non trouvés: 365


In [24]:
import pandas as pd

linked_df = pd.DataFrame(linked)

# sauvegarde
linked_df.to_csv("../kg_artifacts/entity_alignment_table.csv", index=False)
print("Sauvegardé → kg_artifacts/entity_alignment_table.csv")

# inspecte les non trouvés par type
print("\nNon trouvés par type:")
not_found_df = linked_df[linked_df["wikidata_uri"].isna()]
print(not_found_df["entity_type"].value_counts())

print("\nExemples non trouvés (Driver):")
print(not_found_df[not_found_df["entity_type"] == "Driver"]["entity_text"].head(20).tolist())

Sauvegardé → kg_artifacts/entity_alignment_table.csv

Non trouvés par type:
entity_type
Driver       282
GrandPrix     83
Name: count, dtype: int64

Exemples non trouvés (Driver):
['- van de Burgt', 'Agencia Cubana de Noticias', 'Al Hashmi', 'Andrea Margutti Trophy - OKJ[clarification', 'Andrea de Cesaris.[53][54', 'Antonio Fuoco.[27][28', 'Appendix 1 - ^ "', 'Arrow McLaren Rounds', 'Arrow Schmidt Peterson', 'Arrow Schmidt Peterson Motorsports', 'Aston Martin F1.com', 'Atlas F1', 'Audi Sport ABT Schaeffler', 'Austria 2023 - Championship', 'Autodrom Igora Drive', 'Avio Costruzioni Ferrari', 'Axalta Racing', 'Ayrton Senna Alain', 'Azerbaijan Grands', 'BBC Panorama']


In [25]:
found_df = linked_df[linked_df["wikidata_uri"].notna()]

print("Trouvés par type:")
print(found_df["entity_type"].value_counts())

print("\nExemples trouvés (Driver, confidence=1.0):")
print(found_df[
    (found_df["entity_type"] == "Driver") & 
    (found_df["confidence"] == 1.0)
]["entity_text"].head(20).tolist())

Trouvés par type:
entity_type
Driver       415
GrandPrix     50
Team          10
Name: count, dtype: int64

Exemples trouvés (Driver, confidence=1.0):
['Achille Varzi', 'Adrian Newey', 'Alan Jones', 'Alberto Ascari', 'Alessandro Pier Guidi', 'Alex Lloyd', 'Alexander Albon', 'Alexander Rossi', 'Alfa Romeo', 'Andrea Kimi Antonelli', 'Andrea de Adamich', 'Anthony Kumpen', 'Antonio Fuoco', 'Antonio Giovinazzi', 'Ayrton Senna', 'Brendon Leigh', 'Brian Barnhart', 'Bruno Giacomelli', 'Bruno Senna', 'Carlos Reutemann']


In [26]:
from src.kg.alignment_builder import build_alignment_graph, get_aligned_qids

alignment_graph = build_alignment_graph(linked)

print(f"Triples d'alignement: {len(alignment_graph)}")

qids = get_aligned_qids(linked, min_confidence=0.6)
print(f"QIDs pour expansion: {len(qids)}")

Triples d'alignement: 949
QIDs pour expansion: 475


In [27]:
alignment_graph.serialize("../kg_artifacts/alignment.ttl", format="turtle")
print("Sauvegardé → kg_artifacts/alignment.ttl")

Sauvegardé → kg_artifacts/alignment.ttl


In [29]:
import requests

url = "https://query.wikidata.org/sparql"
headers = {
    "User-Agent": "F1-KG-Project/1.0 (student project; paula@example.com)",
    "Accept": "application/sparql-results+json",
}

query = """
SELECT ?p ?o WHERE {
  wd:Q9673 ?p ?o .
}
LIMIT 10
"""

response = requests.get(url, params={"query": query, "format": "json"}, headers=headers, timeout=30)
print("Status:", response.status_code)
print("Response:", response.text[:500])

Status: 200
Response: {
  "head" : {
    "vars" : [ "p", "o" ]
  },
  "results" : {
    "bindings" : [ {
      "p" : {
        "type" : "uri",
        "value" : "http://schema.org/version"
      },
      "o" : {
        "datatype" : "http://www.w3.org/2001/XMLSchema#integer",
        "type" : "literal",
        "value" : "2474909640"
      }
    }, {
      "p" : {
        "type" : "uri",
        "value" : "http://schema.org/dateModified"
      },
      "o" : {
        "datatype" : "http://www.w3.org/2001/XMLSchema#da


In [32]:
print("Status:", response.status_code)
print("Response:", response.text[:200])

Status: 502
Response: <html>
<head><title>502 Bad Gateway</title></head>
<body>
<center><h1>502 Bad Gateway</h1></center>
<hr><center>nginx/1.18.0</center>
</body>
</html>



In [33]:
import time
time.sleep(30)

query = """
SELECT ?p ?o WHERE {
  wd:Q9673 ?p ?o .
  FILTER(STRSTARTS(STR(?p), "http://www.wikidata.org/prop/direct/"))
}
LIMIT 20
"""

response = requests.get(url, params={"query": query, "format": "json"}, headers=headers, timeout=30)
print("Status:", response.status_code)

if response.status_code == 200:
    bindings = response.json()["results"]["bindings"]
    for row in bindings:
        pred = row["p"]["value"]
        obj  = row["o"]["value"]
        print(f"{pred.split('/')[-1]:10} | {obj[:60]}")

Status: 200
P18        | http://commons.wikimedia.org/wiki/Special:FilePath/Prime%20M
P19        | http://www.wikidata.org/entity/Q19795
P21        | http://www.wikidata.org/entity/Q6581097
P22        | http://www.wikidata.org/entity/Q134614524
P27        | http://www.wikidata.org/entity/Q145
P31        | http://www.wikidata.org/entity/Q5
P54        | http://www.wikidata.org/entity/Q169898
P54        | http://www.wikidata.org/entity/Q172030
P54        | http://www.wikidata.org/entity/Q172721
P69        | http://www.wikidata.org/entity/Q7743376
P97        | http://www.wikidata.org/entity/Q102083
P97        | http://www.wikidata.org/entity/Q11105296
P106       | http://www.wikidata.org/entity/Q10349745
P106       | http://www.wikidata.org/entity/Q10841764
P109       | http://commons.wikimedia.org/wiki/Special:FilePath/Signature
P140       | http://www.wikidata.org/entity/Q9592
P166       | http://www.wikidata.org/entity/Q680221
P166       | http://www.wikidata.org/entity/Q1061233
P166   

In [35]:
from src.kg.sparql_expander import expand_one_hop

for qid in ["Q9673", "Q169898", "Q193662"]:
    raw = expand_one_hop(qid, limit=50)
    print(f"{qid} → {len(raw)} triples")

Q9673 → 8 triples
Q169898 → 14 triples
Q193662 → 4 triples


In [36]:
from src.kg.sparql_expander import expand_all

# saisons présentes dans le KB
season_years = entity_df[entity_df["entity_type"] == "Season"]["entity_text"].unique().tolist()
print(f"Saisons: {season_years}")

expansion_graph = expand_all(
    qids=qids,
    season_years=season_years,
    one_hop_limit=200,
    season_limit=1000,
    delay=1.0,
)

print(f"\nTotal triples expansion: {len(expansion_graph)}")

Saisons: ['1351', '1892', '1929', '1931', '1932', '1933', '1934', '1935', '1937', '1938', '1939', '1940', '1943', '1944', '1945', '1947', '1948', '1949', '1950', '1951', '1952', '1953', '1954', '1955', '1956', '1957', '1958', '1959', '1960', '1961', '1962', '1963', '1964', '1965', '1966', '1967', '1968', '1969', '1970', '1971', '1972', '1973', '1974', '1975', '1976', '1977', '1978', '1979', '1980', '1981', '1982', '1983', '1984', '1985', '1986', '1987', '1988', '1989', '1990', '1991', '1992', '1993', '1994', '1995', '1996', '1997', '1998', '1999', '2000', '2001', '2002', '2003', '2004', '2005', '2006', '2007', '2008', '2009', '2010', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024', '2025', '2026', '2027', '2030']
[sparql_expander] Expansion 1-hop pour 475 entités...
  50/475 entités traitées — 310 triples
  100/475 entités traitées — 655 triples
  150/475 entités traitées — 950 triples
  200/475 entités traitées — 1262 trip

In [37]:
# filtre les saisons F1 valides (1950 à aujourd'hui)
season_years = [
    y for y in entity_df[entity_df["entity_type"] == "Season"]["entity_text"].unique().tolist()
    if y.isdigit() and 1950 <= int(y) <= 2026
]
print(f"Saisons F1 valides: {season_years}")

Saisons F1 valides: ['1950', '1951', '1952', '1953', '1954', '1955', '1956', '1957', '1958', '1959', '1960', '1961', '1962', '1963', '1964', '1965', '1966', '1967', '1968', '1969', '1970', '1971', '1972', '1973', '1974', '1975', '1976', '1977', '1978', '1979', '1980', '1981', '1982', '1983', '1984', '1985', '1986', '1987', '1988', '1989', '1990', '1991', '1992', '1993', '1994', '1995', '1996', '1997', '1998', '1999', '2000', '2001', '2002', '2003', '2004', '2005', '2006', '2007', '2008', '2009', '2010', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024', '2025', '2026']


In [38]:
import requests, time

url = "https://query.wikidata.org/sparql"
headers = {
    "User-Agent": "F1-KG-Project/1.0 (student project; paula@example.com)",
    "Accept": "application/sparql-results+json",
}

query = """
SELECT ?race ?p ?o WHERE {
  ?race wdt:P31 wd:Q941966 .
  ?race wdt:P585 ?date .
  FILTER(YEAR(?date) = 2024)
  ?race ?p ?o .
  FILTER(STRSTARTS(STR(?p), "http://www.wikidata.org/prop/direct/"))
}
LIMIT 50
"""

response = requests.get(url, params={"query": query, "format": "json"}, headers=headers, timeout=30)
print("Status:", response.status_code)
if response.status_code == 200:
    bindings = response.json()["results"]["bindings"]
    print(f"Résultats: {len(bindings)}")
    for row in bindings[:5]:
        print(row["race"]["value"].split("/")[-1], "|", row["p"]["value"].split("/")[-1], "|", row["o"]["value"][:50])

Status: 200
Résultats: 0


In [39]:
# cherchons les prédicats du Grand Prix de Monaco 2024 (Q123482134)
query = """
SELECT ?p ?o WHERE {
  wd:Q123482134 ?p ?o .
  FILTER(STRSTARTS(STR(?p), "http://www.wikidata.org/prop/direct/"))
}
LIMIT 30
"""

response = requests.get(url, params={"query": query, "format": "json"}, headers=headers, timeout=30)
print("Status:", response.status_code)
if response.status_code == 200:
    bindings = response.json()["results"]["bindings"]
    for row in bindings:
        print(row["p"]["value"].split("/")[-1], "|", row["o"]["value"][:60])

Status: 200
P31 | http://www.wikidata.org/entity/Q3305213
P136 | http://www.wikidata.org/entity/Q1047337
P170 | http://www.wikidata.org/entity/Q2048944
P186 | http://www.wikidata.org/entity/Q296955
P186 | http://www.wikidata.org/entity/Q12321255
P350 | 61917
P571 | 1890-01-01T00:00:00Z
P1476 | Twee mannen bekijken een schilderij aan de wand
P2048 | 46.5
P2049 | 73.5


In [43]:
import time
time.sleep(30)

# Qiji F1 Grand Prix = Q1968561 (Formula One race)
query_search = """
SELECT ?race ?label WHERE {
  ?race wdt:P31 wd:Q1968561 .
  ?race rdfs:label ?label .
  FILTER(LANG(?label) = "en")
  FILTER(CONTAINS(?label, "2024"))
}
LIMIT 10
"""

response = requests.get(url, params={"query": query_search, "format": "json"}, headers=headers, timeout=60)
print("Status:", response.status_code)
if response.status_code == 200:
    bindings = response.json()["results"]["bindings"]
    print(f"Résultats: {len(bindings)}")
    for row in bindings[:5]:
        print(row["race"]["value"].split("/")[-1], "|", row["label"]["value"])

Status: 200
Résultats: 0


In [44]:
time.sleep(10)

query_search = """
SELECT ?race ?type ?label WHERE {
  ?race rdfs:label "2024 Monaco Grand Prix"@en .
  ?race wdt:P31 ?type .
  ?race rdfs:label ?label .
  FILTER(LANG(?label) = "en")
}
LIMIT 5
"""

response = requests.get(url, params={"query": query_search, "format": "json"}, headers=headers, timeout=60)
print("Status:", response.status_code)
if response.status_code == 200:
    bindings = response.json()["results"]["bindings"]
    print(f"Résultats: {len(bindings)}")
    for row in bindings:
        print(row["race"]["value"].split("/")[-1], "|", row["type"]["value"].split("/")[-1], "|", row["label"]["value"])

Status: 200
Résultats: 2
Q115910301 | Q9102 | 2024 Monaco Grand Prix
Q115910301 | Q11924610 | 2024 Monaco Grand Prix


In [46]:
time.sleep(10)

# on regarde tous les prédicats de Q115910301 (2024 Monaco Grand Prix)
query = """
SELECT ?p ?o WHERE {
  wd:Q115910301 ?p ?o .
  FILTER(STRSTARTS(STR(?p), "http://www.wikidata.org/prop/direct/"))
}
LIMIT 30
"""

response = requests.get(url, params={"query": query, "format": "json"}, headers=headers, timeout=60)
print("Status:", response.status_code)
if response.status_code == 200:
    bindings = response.json()["results"]["bindings"]
    print(f"Résultats: {len(bindings)}")
    for row in bindings:
        print(row["p"]["value"].split("/")[-1], "|", row["o"]["value"][:60])

Status: 200
Résultats: 17
P17 | http://www.wikidata.org/entity/Q235
P18 | http://commons.wikimedia.org/wiki/Special:FilePath/Circuit%2
P31 | http://www.wikidata.org/entity/Q9102
P31 | http://www.wikidata.org/entity/Q11924610
P276 | http://www.wikidata.org/entity/Q45240
P276 | http://www.wikidata.org/entity/Q171400
P361 | http://www.wikidata.org/entity/Q113628886
P580 | 2024-05-24T00:00:00Z
P582 | 2024-05-26T00:00:00Z
P641 | http://www.wikidata.org/entity/Q1968
P641 | http://www.wikidata.org/entity/Q2705092
P1346 | http://www.wikidata.org/entity/Q169898
P1346 | http://www.wikidata.org/entity/Q17541912
P1705 | Formula 1 Grand Prix de Monaco 2024
P2671 | /g/11l7996309
P3157 | 78
P3450 | http://www.wikidata.org/entity/Q9102


In [47]:
time.sleep(10)

from src.kg.sparql_expander import expand_f1_season, triples_to_rdf_graph

raw = expand_f1_season("2024", limit=500)
print(f"2024 → {len(raw)} triples")
for t in raw[:5]:
    print(t)

2024 → 12 triples
('http://www.wikidata.org/entity/Q115910301', 'http://www.wikidata.org/prop/direct/P17', ('uri', 'http://www.wikidata.org/entity/Q235'))
('http://www.wikidata.org/entity/Q115910301', 'http://www.wikidata.org/prop/direct/P31', ('uri', 'http://www.wikidata.org/entity/Q9102'))
('http://www.wikidata.org/entity/Q115910301', 'http://www.wikidata.org/prop/direct/P31', ('uri', 'http://www.wikidata.org/entity/Q11924610'))
('http://www.wikidata.org/entity/Q115910301', 'http://www.wikidata.org/prop/direct/P276', ('uri', 'http://www.wikidata.org/entity/Q45240'))
('http://www.wikidata.org/entity/Q115910301', 'http://www.wikidata.org/prop/direct/P276', ('uri', 'http://www.wikidata.org/entity/Q171400'))


In [48]:
time.sleep(10)

# expansion 2-hop : depuis Ferrari, qui a gagné quoi ?
query = """
SELECT ?s ?p ?o WHERE {
  ?s wdt:P54 wd:Q169898 .
  ?s ?p ?o .
  FILTER(STRSTARTS(STR(?p), "http://www.wikidata.org/prop/direct/"))
}
LIMIT 200
"""

response = requests.get(url, params={"query": query, "format": "json"}, headers=headers, timeout=60)
print("Status:", response.status_code)
if response.status_code == 200:
    bindings = response.json()["results"]["bindings"]
    print(f"Résultats: {len(bindings)}")
    for row in bindings[:5]:
        print(row["s"]["value"].split("/")[-1], "|", row["p"]["value"].split("/")[-1], "|", row["o"]["value"][:50])

Status: 200
Résultats: 200
Q2040 | P18 | http://commons.wikimedia.org/wiki/Special:FilePath
Q2040 | P19 | http://www.wikidata.org/entity/Q495
Q2040 | P20 | http://www.wikidata.org/entity/Q165090
Q2040 | P21 | http://www.wikidata.org/entity/Q6581097
Q2040 | P27 | http://www.wikidata.org/entity/Q38


In [49]:
from src.kg.sparql_expander import expand_by_team, triples_to_rdf_graph

team_qids = [
    "Q98930155",  # Alpine
    "Q104620704", # Aston Martin
    "Q120756696", # Audi
    "Q131323328", # Cadillac
    "Q169898",    # Ferrari
    "Q16300222",  # Haas
    "Q172030",    # McLaren
    "Q172721",    # Mercedes
    "Q173663",    # Red Bull
    "Q171337",    # Williams
]

total = 0
for qid in team_qids:
    raw = expand_by_team(qid, limit=500)
    total += len(raw)
    print(f"{qid} → {len(raw)} triples")
    time.sleep(1)

print(f"\nTotal: {total} triples")

Q98930155 → 31 triples
Q104620704 → 41 triples
Q120756696 → 15 triples
Q131323328 → 0 triples
Q169898 → 87 triples
Q16300222 → 96 triples
Q172030 → 109 triples
Q172721 → 96 triples
Q173663 → 88 triples
Q171337 → 83 triples

Total: 646 triples


In [50]:
time.sleep(10)

# tous les drivers F1 et leurs propriétés en une seule requête
query = """
SELECT ?driver ?p ?o WHERE {
  ?driver wdt:P31 wd:Q5 .
  ?driver wdt:P106 wd:Q10843402 .
  ?driver ?p ?o .
  FILTER(STRSTARTS(STR(?p), "http://www.wikidata.org/prop/direct/"))
}
LIMIT 5000
"""

response = requests.get(url, params={"query": query, "format": "json"}, headers=headers, timeout=120)
print("Status:", response.status_code)
if response.status_code == 200:
    bindings = response.json()["results"]["bindings"]
    print(f"Résultats: {len(bindings)}")

Status: 200
Résultats: 5000


In [51]:
time.sleep(10)

all_bindings = []

for offset in range(0, 50000, 5000):
    query = f"""
    SELECT ?driver ?p ?o WHERE {{
      ?driver wdt:P31 wd:Q5 .
      ?driver wdt:P106 wd:Q10843402 .
      ?driver ?p ?o .
      FILTER(STRSTARTS(STR(?p), "http://www.wikidata.org/prop/direct/"))
    }}
    LIMIT 5000
    OFFSET {offset}
    """
    
    response = requests.get(url, params={"query": query, "format": "json"}, headers=headers, timeout=120)
    
    if response.status_code != 200:
        print(f"Erreur à offset {offset}: {response.status_code}")
        break
    
    bindings = response.json()["results"]["bindings"]
    all_bindings.extend(bindings)
    print(f"Offset {offset} → {len(bindings)} résultats (total: {len(all_bindings)})")
    
    if len(bindings) < 5000:
        print("Fin des résultats")
        break
    
    time.sleep(2)

print(f"\nTotal bindings: {len(all_bindings)}")

Offset 0 → 5000 résultats (total: 5000)
Offset 5000 → 5000 résultats (total: 10000)
Offset 10000 → 5000 résultats (total: 15000)
Offset 15000 → 5000 résultats (total: 20000)
Offset 20000 → 5000 résultats (total: 25000)
Offset 25000 → 5000 résultats (total: 30000)
Offset 30000 → 5000 résultats (total: 35000)
Offset 35000 → 5000 résultats (total: 40000)
Offset 40000 → 5000 résultats (total: 45000)
Offset 45000 → 5000 résultats (total: 50000)

Total bindings: 50000


In [52]:
from rdflib import Graph, URIRef, Literal

def bindings_to_graph(bindings, subject_key="driver"):
    graph = Graph()
    
    for row in bindings:
        subj     = row.get(subject_key, {}).get("value", "")
        pred     = row.get("p", {}).get("value", "")
        obj_info = row.get("o", {})
        obj_val  = obj_info.get("value", "")
        obj_type = obj_info.get("type", "")
        
        if not subj or pred not in ALLOWED_PREDICATES:
            continue
        
        subject   = URIRef(subj)
        predicate = URIRef(pred)
        
        if obj_type == "uri":
            graph.add((subject, predicate, URIRef(obj_val)))
        elif obj_type == "literal":
            lang     = obj_info.get("xml:lang")
            datatype = obj_info.get("datatype")
            if lang:
                graph.add((subject, predicate, Literal(obj_val, lang=lang)))
            elif datatype:
                graph.add((subject, predicate, Literal(obj_val, datatype=URIRef(datatype))))
            else:
                graph.add((subject, predicate, Literal(obj_val)))
    
    return graph

ALLOWED_PREDICATES = {
    "http://www.wikidata.org/prop/direct/P31",
    "http://www.wikidata.org/prop/direct/P21",
    "http://www.wikidata.org/prop/direct/P27",
    "http://www.wikidata.org/prop/direct/P569",
    "http://www.wikidata.org/prop/direct/P19",
    "http://www.wikidata.org/prop/direct/P54",
    "http://www.wikidata.org/prop/direct/P1344",
    "http://www.wikidata.org/prop/direct/P1346",
    "http://www.wikidata.org/prop/direct/P2522",
    "http://www.wikidata.org/prop/direct/P17",
    "http://www.wikidata.org/prop/direct/P361",
    "http://www.wikidata.org/prop/direct/P580",
    "http://www.wikidata.org/prop/direct/P582",
    "http://www.wikidata.org/prop/direct/P159",
    "http://www.wikidata.org/prop/direct/P571",
    "http://www.wikidata.org/prop/direct/P276",
    "http://www.wikidata.org/prop/direct/P641",
}

drivers_graph = bindings_to_graph(all_bindings, subject_key="driver")
print(f"Triples drivers: {len(drivers_graph)}")

Triples drivers: 13171


In [53]:
# merge tout ce qu'on a
from src.kg.sparql_expander import triples_to_rdf_graph

# 1. graph initial (2441 triples)
# 2. alignment graph (949 triples)  
# 3. expansion 1-hop (2776 triples)
# 4. drivers graph (13171 triples)

full_expansion = drivers_graph + expansion_graph

print(f"Expansion totale: {len(full_expansion)} triples")

# merge avec le graph initial et alignment
full_kb = graph + alignment_graph + full_expansion
print(f"KB complète: {len(full_kb)} triples")

Expansion totale: 15947 triples
KB complète: 19337 triples


In [54]:
time.sleep(10)

all_team_bindings = []

for offset in range(0, 20000, 5000):
    query = f"""
    SELECT ?team ?p ?o WHERE {{
      ?team wdt:P31 wd:Q3354859 .
      ?team ?p ?o .
      FILTER(STRSTARTS(STR(?p), "http://www.wikidata.org/prop/direct/"))
    }}
    LIMIT 5000
    OFFSET {offset}
    """
    
    response = requests.get(url, params={"query": query, "format": "json"}, headers=headers, timeout=120)
    
    if response.status_code != 200:
        print(f"Erreur à offset {offset}: {response.status_code}")
        break
    
    bindings = response.json()["results"]["bindings"]
    all_team_bindings.extend(bindings)
    print(f"Offset {offset} → {len(bindings)} résultats (total: {len(all_team_bindings)})")
    
    if len(bindings) < 5000:
        print("Fin des résultats")
        break
    
    time.sleep(2)

teams_graph = bin

Offset 0 → 1815 résultats (total: 1815)
Fin des résultats


In [55]:
time.sleep(10)

# circuits de course (Q1777138)
all_circuit_bindings = []

for offset in range(0, 20000, 5000):
    query = f"""
    SELECT ?circuit ?p ?o WHERE {{
      ?circuit wdt:P31 wd:Q1777138 .
      ?circuit ?p ?o .
      FILTER(STRSTARTS(STR(?p), "http://www.wikidata.org/prop/direct/"))
    }}
    LIMIT 5000
    OFFSET {offset}
    """
    
    response = requests.get(url, params={"query": query, "format": "json"}, headers=headers, timeout=120)
    
    if response.status_code != 200:
        print(f"Erreur à offset {offset}: {response.status_code}")
        break
    
    bindings = response.json()["results"]["bindings"]
    all_circuit_bindings.extend(bindings)
    print(f"Offset {offset} → {len(bindings)} résultats (total: {len(all_circuit_bindings)})")
    
    if len(bindings) < 5000:
        print("Fin des résultats")
        break
    
    time.sleep(2)

circuits_graph = bindings_to_graph(all_circuit_bindings, subject_key="circuit")
print(f"Triples circuits: {len(circuits_graph)}")

Offset 0 → 977 résultats (total: 977)
Fin des résultats
Triples circuits: 333


In [56]:
time.sleep(10)

# saisons F1 (Q1539532 = Formula One season)
all_season_bindings = []

for offset in range(0, 20000, 5000):
    query = f"""
    SELECT ?season ?p ?o WHERE {{
      ?season wdt:P31 wd:Q1539532 .
      ?season ?p ?o .
      FILTER(STRSTARTS(STR(?p), "http://www.wikidata.org/prop/direct/"))
    }}
    LIMIT 5000
    OFFSET {offset}
    """
    
    response = requests.get(url, params={"query": query, "format": "json"}, headers=headers, timeout=120)
    
    if response.status_code != 200:
        print(f"Erreur à offset {offset}: {response.status_code}")
        break
    
    bindings = response.json()["results"]["bindings"]
    all_season_bindings.extend(bindings)
    print(f"Offset {offset} → {len(bindings)} résultats (total: {len(all_season_bindings)})")
    
    if len(bindings) < 5000:
        print("Fin des résultats")
        break
    
    time.sleep(2)

seasons_graph = bindings_to_graph(all_season_bindings, subject_key="season")
print(f"Triples saisons: {len(seasons_graph)}")

Offset 0 → 5000 résultats (total: 5000)
Erreur à offset 5000: 502
Triples saisons: 2497


In [57]:
full_expansion = drivers_graph + teams_graph + circuits_graph + seasons_graph + expansion_graph

print(f"Expansion totale: {len(full_expansion)} triples")

full_kb = graph + alignment_graph + full_expansion
print(f"KB complète: {len(full_kb)} triples")

AttributeError: 'builtin_function_or_method' object has no attribute 'namespaces'

In [59]:
print(type(drivers_graph))
print(type(teams_graph))
print(type(circuits_graph))
print(type(seasons_graph))
print(type(graph))
print(type(alignment_graph))

<class 'rdflib.graph.Graph'>
<class 'builtin_function_or_method'>
<class 'rdflib.graph.Graph'>
<class 'rdflib.graph.Graph'>
<class 'rdflib.graph.Graph'>
<class 'rdflib.graph.Graph'>


In [60]:
teams_graph = bindings_to_graph(all_team_bindings, subject_key="team")
print(f"Triples équipes: {len(teams_graph)}")

Triples équipes: 129


In [61]:
full_expansion = drivers_graph + teams_graph + circuits_graph + seasons_graph

print(f"Expansion totale: {len(full_expansion)} triples")

full_kb = graph + alignment_graph + full_expansion
print(f"KB complète: {len(full_kb)} triples")

Expansion totale: 16130 triples
KB complète: 19520 triples


In [62]:
time.sleep(10)

# Grand Prix F1 (Q9102 = Formula One race)
all_gp_bindings = []

for offset in range(0, 50000, 5000):
    query = f"""
    SELECT ?gp ?p ?o WHERE {{
      ?gp wdt:P31 wd:Q9102 .
      ?gp ?p ?o .
      FILTER(STRSTARTS(STR(?p), "http://www.wikidata.org/prop/direct/"))
    }}
    LIMIT 5000
    OFFSET {offset}
    """
    
    response = requests.get(url, params={"query": query, "format": "json"}, headers=headers, timeout=120)
    
    if response.status_code != 200:
        print(f"Erreur à offset {offset}: {response.status_code}")
        break
    
    bindings = response.json()["results"]["bindings"]
    all_gp_bindings.extend(bindings)
    print(f"Offset {offset} → {len(bindings)} résultats (total: {len(all_gp_bindings)})")
    
    if len(bindings) < 5000:
        print("Fin des résultats")
        break
    
    time.sleep(3)

gp_graph = bindings_to_graph(all_gp_bindings, subject_key="gp")
print(f"\nTriples Grand Prix: {len(gp_graph)}")

Offset 0 → 1627 résultats (total: 1627)
Fin des résultats

Triples Grand Prix: 723


In [63]:
# quels prédicats dans all_gp_bindings sont filtrés ?
from collections import Counter

pred_counter = Counter()
for row in all_gp_bindings:
    pred = row.get("p", {}).get("value", "").split("/")[-1]
    pred_counter[pred] += 1

print("Top 20 prédicats dans les Grand Prix:")
for pred, count in pred_counter.most_common(20):
    in_allowed = f"http://www.wikidata.org/prop/direct/{pred}" in ALLOWED_PREDICATES
    print(f"  {'✅' if in_allowed else '❌'} {pred:10} | {count} occurrences")

Top 20 prédicats dans les Grand Prix:
  ✅ P31        | 171 occurrences
  ✅ P1346      | 161 occurrences
  ✅ P276       | 137 occurrences
  ✅ P641       | 84 occurrences
  ✅ P17        | 83 occurrences
  ❌ P3157      | 80 occurrences
  ❌ P3764      | 80 occurrences
  ❌ P585       | 78 occurrences
  ❌ P625       | 78 occurrences
  ❌ P5053      | 78 occurrences
  ✅ P361       | 73 occurrences
  ❌ P3450      | 72 occurrences
  ❌ P646       | 71 occurrences
  ❌ P1448      | 68 occurrences
  ❌ P2283      | 67 occurrences
  ❌ P6806      | 67 occurrences
  ❌ P373       | 47 occurrences
  ❌ P1705      | 46 occurrences
  ❌ P710       | 23 occurrences
  ❌ P18        | 18 occurrences


In [64]:
# recrée tous les graphs avec les nouveaux prédicats
drivers_graph   = bindings_to_graph(all_bindings,        subject_key="driver")
teams_graph     = bindings_to_graph(all_team_bindings,   subject_key="team")
circuits_graph  = bindings_to_graph(all_circuit_bindings, subject_key="circuit")
seasons_graph   = bindings_to_graph(all_season_bindings, subject_key="season")
gp_graph        = bindings_to_graph(all_gp_bindings,     subject_key="gp")

full_expansion = drivers_graph + teams_graph + circuits_graph + seasons_graph + gp_graph
full_kb = graph + alignment_graph + full_expansion

print(f"Drivers  : {len(drivers_graph)}")
print(f"Teams    : {len(teams_graph)}")
print(f"Circuits : {len(circuits_graph)}")
print(f"Saisons  : {len(seasons_graph)}")
print(f"Grand Prix: {len(gp_graph)}")
print(f"\nExpansion totale: {len(full_expansion)}")
print(f"KB complète     : {len(full_kb)}")

Drivers  : 13171
Teams    : 129
Circuits : 333
Saisons  : 2497
Grand Prix: 723

Expansion totale: 16853
KB complète     : 20243


In [65]:
time.sleep(10)

# on continue la pagination des drivers à partir de offset 50000
all_bindings_2 = []

for offset in range(50000, 150000, 5000):
    query = f"""
    SELECT ?driver ?p ?o WHERE {{
      ?driver wdt:P31 wd:Q5 .
      ?driver wdt:P106 wd:Q10843402 .
      ?driver ?p ?o .
      FILTER(STRSTARTS(STR(?p), "http://www.wikidata.org/prop/direct/"))
    }}
    LIMIT 5000
    OFFSET {offset}
    """
    
    response = requests.get(url, params={"query": query, "format": "json"}, headers=headers, timeout=120)
    
    if response.status_code != 200:
        print(f"Erreur à offset {offset}: {response.status_code}")
        break
    
    bindings = response.json()["results"]["bindings"]
    all_bindings_2.extend(bindings)
    print(f"Offset {offset} → {len(bindings)} résultats (total: {len(all_bindings_2)})")
    
    if len(bindings) < 5000:
        print("Fin des résultats")
        break
    
    time.sleep(3)

drivers_graph_2 = bindings_to_graph(all_bindings_2, subject_key="driver")
print(f"\nTriples drivers supplémentaires: {len(drivers_graph_2)}")

Offset 50000 → 5000 résultats (total: 5000)
Offset 55000 → 5000 résultats (total: 10000)
Offset 60000 → 5000 résultats (total: 15000)
Offset 65000 → 5000 résultats (total: 20000)
Offset 70000 → 5000 résultats (total: 25000)
Offset 75000 → 5000 résultats (total: 30000)
Offset 80000 → 5000 résultats (total: 35000)
Offset 85000 → 5000 résultats (total: 40000)
Offset 90000 → 5000 résultats (total: 45000)
Offset 95000 → 5000 résultats (total: 50000)
Offset 100000 → 5000 résultats (total: 55000)
Offset 105000 → 5000 résultats (total: 60000)
Offset 110000 → 5000 résultats (total: 65000)
Offset 115000 → 5000 résultats (total: 70000)
Offset 120000 → 5000 résultats (total: 75000)
Offset 125000 → 5000 résultats (total: 80000)
Offset 130000 → 5000 résultats (total: 85000)
Offset 135000 → 5000 résultats (total: 90000)
Offset 140000 → 5000 résultats (total: 95000)
Offset 145000 → 5000 résultats (total: 100000)

Triples drivers supplémentaires: 26726


In [66]:
full_expansion = drivers_graph + drivers_graph_2 + teams_graph + circuits_graph + seasons_graph + gp_graph

full_kb = graph + alignment_graph + full_expansion

print(f"Drivers 1  : {len(drivers_graph)}")
print(f"Drivers 2  : {len(drivers_graph_2)}")
print(f"Teams      : {len(teams_graph)}")
print(f"Circuits   : {len(circuits_graph)}")
print(f"Saisons    : {len(seasons_graph)}")
print(f"Grand Prix : {len(gp_graph)}")
print(f"\nExpansion totale : {len(full_expansion)}")
print(f"KB complète      : {len(full_kb)}")

Drivers 1  : 13171
Drivers 2  : 26726
Teams      : 129
Circuits   : 333
Saisons    : 2497
Grand Prix : 723

Expansion totale : 38829
KB complète      : 42219


In [67]:
time.sleep(10)

# relance équipes avec les nouveaux ALLOWED_PREDICATES
all_team_bindings = []

for offset in range(0, 20000, 5000):
    query = f"""
    SELECT ?team ?p ?o WHERE {{
      ?team wdt:P31 wd:Q3354859 .
      ?team ?p ?o .
      FILTER(STRSTARTS(STR(?p), "http://www.wikidata.org/prop/direct/"))
    }}
    LIMIT 5000
    OFFSET {offset}
    """
    
    response = requests.get(url, params={"query": query, "format": "json"}, headers=headers, timeout=120)
    
    if response.status_code != 200:
        print(f"Erreur à offset {offset}: {response.status_code}")
        break
    
    bindings = response.json()["results"]["bindings"]
    all_team_bindings.extend(bindings)
    print(f"Offset {offset} → {len(bindings)} résultats (total: {len(all_team_bindings)})")
    
    if len(bindings) < 5000:
        print("Fin des résultats")
        break
    
    time.sleep(2)

teams_graph = bindings_to_graph(all_team_bindings, subject_key="team")
print(f"Triples équipes: {len(teams_graph)}")

Offset 0 → 1815 résultats (total: 1815)
Fin des résultats
Triples équipes: 129


In [68]:
time.sleep(10)

# pilotes de course en général (Q13029807 = racing driver, plus large que F1)
all_moto_bindings = []

for offset in range(0, 30000, 5000):
    query = f"""
    SELECT ?driver ?p ?o WHERE {{
      ?driver wdt:P31 wd:Q5 .
      ?driver wdt:P106 wd:Q13029807 .
      ?driver ?p ?o .
      FILTER(STRSTARTS(STR(?p), "http://www.wikidata.org/prop/direct/"))
    }}
    LIMIT 5000
    OFFSET {offset}
    """
    
    response = requests.get(url, params={"query": query, "format": "json"}, headers=headers, timeout=120)
    
    if response.status_code != 200:
        print(f"Erreur à offset {offset}: {response.status_code}")
        break
    
    bindings = response.json()["results"]["bindings"]
    all_moto_bindings.extend(bindings)
    print(f"Offset {offset} → {len(bindings)} résultats (total: {len(all_moto_bindings)})")
    
    if len(bindings) < 5000:
        print("Fin des résultats")
        break
    
    time.sleep(3)

moto_graph = bindings_to_graph(all_moto_bindings, subject_key="driver")
print(f"\nTriples motorsport: {len(moto_graph)}")

Offset 0 → 0 résultats (total: 0)
Fin des résultats

Triples motorsport: 0


In [70]:
time.sleep(10)

# vérifie d'abord le bon QID pour racing driver
query = """
SELECT ?type ?label WHERE {
  ?type rdfs:label "racing driver"@en .
  ?type wdt:P31 wd:Q16334295 .
}
LIMIT 5
"""

response = requests.get(url, params={"query": query, "format": "json"}, headers=headers, timeout=60)
if response.status_code == 200:
    bindings = response.json()["results"]["bindings"]
    for row in bindings:
        print(row["type"]["value"].split("/")[-1], "|", row["label"]["value"])
else:
    # essai direct avec Q806798
    query2 = """
    SELECT ?driver ?p ?o WHERE {
      ?driver wdt:P31 wd:Q5 .
      ?driver wdt:P106 wd:Q806798 .
      ?driver ?p ?o .
      FILTER(STRSTARTS(STR(?p), "http://www.wikidata.org/prop/direct/"))
    }
    LIMIT 10
    """
    response2 = requests.get(url, params={"query": query2, "format": "json"}, headers=headers, timeout=60)
    print("Status:", response2.status_code)
    if response2.status_code == 200:
        b = response2.json()["results"]["bindings"]
        print(f"Résultats: {len(b)}")

In [71]:
# merge final
full_expansion = drivers_graph + drivers_graph_2 + teams_graph + circuits_graph + seasons_graph + gp_graph
full_kb = graph + alignment_graph + full_expansion

# sauvegarde
full_expansion.serialize("../kg_artifacts/expanded.nt", format="nt")
full_kb.serialize("../kg_artifacts/full_kb.ttl", format="turtle")

print(f"KB complète: {len(full_kb)} triples → sauvegardé")

# stats
entities   = set(str(s) for s, p, o in full_kb)
predicates = set(str(p) for s, p, o in full_kb)

print(f"Entités uniques    : {len(entities)}")
print(f"Prédicats uniques  : {len(predicates)}")

c:\Users\paula\Documents\Cours\A4\S2\WebData\Project\f1_semantic_web_project\.venv\Lib\site-packages\rdflib\plugins\serializers\nt.py:39: UserWarning: NTSerializer always uses UTF-8 encoding. Given encoding was: None
  warnings.warn(


KB complète: 42219 triples → sauvegardé
Entités uniques    : 7642
Prédicats uniques  : 24


In [72]:
import os

for folder in ["../data", "../kg_artifacts", "../src/kg"]:
    print(f"\n{folder}/")
    if os.path.exists(folder):
        for f in sorted(os.listdir(folder)):
            print(f"  {f}")
    else:
        print("  ❌ n'existe pas")


../data/
  processed
  raw

../kg_artifacts/
  alignment.ttl
  entity_alignment_table.csv
  expanded.nt
  full_kb.ttl
  initial_graph.ttl

../src/kg/
  __init__.py
  __pycache__
  alignment_builder.py
  rdf_builder.py
  schema_utils.py
  sparql_expander.py
  uri_utils.py
  wikidata_linker.py


In [73]:
stats = f"""KB Statistics
=============
Total triples     : {len(full_kb)}
Unique entities   : {len(set(str(s) for s, p, o in full_kb))}
Unique predicates : {len(set(str(p) for s, p, o in full_kb))}
Unique objects    : {len(set(str(o) for s, p, o in full_kb))}

Sources
-------
Initial graph     : 2441 triples
Alignment         : 949 triples
Wikidata expansion: {len(full_kb) - 2441 - 949} triples
"""

with open("../kg_artifacts/kb_stats.txt", "w", encoding="utf-8") as f:
    f.write(stats)

print(stats)

KB Statistics
Total triples     : 42219
Unique entities   : 7642
Unique predicates : 24
Unique objects    : 8125

Sources
-------
Initial graph     : 2441 triples
Alignment         : 949 triples
Wikidata expansion: 38829 triples



In [74]:
import os

for folder in ["../src/kg", "../kg_artifacts"]:
    print(f"\n{folder}/")
    for f in sorted(os.listdir(folder)):
        print(f"  {f}")


../src/kg/
  __init__.py
  __pycache__
  alignment_builder.py
  pipeline.py
  rdf_builder.py
  schema_utils.py
  sparql_expander.py
  uri_utils.py
  wikidata_linker.py

../kg_artifacts/
  alignment.ttl
  entity_alignment_table.csv
  expanded.nt
  full_kb.ttl
  initial_graph.ttl
  kb_stats.txt
  ontology.ttl
